In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("../data/raw/Churn_Modelling.csv")
df.shape

(10000, 14)

In [4]:
# RowNumber, CustomerId, Surname : identifiants sans valeur prédictive
# Gender : exclu pour raison éthique (risque de biais discriminatoire)
df = df.drop(columns=["RowNumber", "CustomerId", "Surname", "Gender"])
df.columns

Index(['CreditScore', 'Geography', 'Age', 'Tenure', 'Balance', 'NumOfProducts',
       'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='str')

In [5]:
X = df.drop(columns=["Exited"])
y = df["Exited"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Vérification de la stratification
print(y_train.value_counts(normalize=True) * 100)
print(y_test.value_counts(normalize=True) * 100)

Exited
0    79.625
1    20.375
Name: proportion, dtype: float64
Exited
0    79.65
1    20.35
Name: proportion, dtype: float64


In [6]:
encoder_geo = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

geo_train_encoded = encoder_geo.fit_transform(X_train[["Geography"]])
geo_test_encoded = encoder_geo.transform(X_test[["Geography"]])

geo_cols = encoder_geo.get_feature_names_out(["Geography"])
geo_train_df = pd.DataFrame(geo_train_encoded, columns=geo_cols, index=X_train.index)
geo_test_df = pd.DataFrame(geo_test_encoded, columns=geo_cols, index=X_test.index)

X_train = pd.concat([X_train.drop(columns=["Geography"]), geo_train_df], axis=1)
X_test = pd.concat([X_test.drop(columns=["Geography"]), geo_test_df], axis=1)

In [7]:
def regrouper_produits(x):
    return "3+" if x >= 3 else str(x)

X_train["NumOfProducts_grp"] = X_train["NumOfProducts"].apply(regrouper_produits)
X_test["NumOfProducts_grp"] = X_test["NumOfProducts"].apply(regrouper_produits)
X_train = X_train.drop(columns=["NumOfProducts"])
X_test = X_test.drop(columns=["NumOfProducts"])

encoder_prod = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
prod_train_encoded = encoder_prod.fit_transform(X_train[["NumOfProducts_grp"]])
prod_test_encoded = encoder_prod.transform(X_test[["NumOfProducts_grp"]])

prod_cols = encoder_prod.get_feature_names_out(["NumOfProducts_grp"])
prod_train_df = pd.DataFrame(prod_train_encoded, columns=prod_cols, index=X_train.index)
prod_test_df = pd.DataFrame(prod_test_encoded, columns=prod_cols, index=X_test.index)

X_train = pd.concat([X_train.drop(columns=["NumOfProducts_grp"]), prod_train_df], axis=1)
X_test = pd.concat([X_test.drop(columns=["NumOfProducts_grp"]), prod_test_df], axis=1)

X_train.columns

Index(['CreditScore', 'Age', 'Tenure', 'Balance', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Geography_France',
       'Geography_Germany', 'Geography_Spain', 'NumOfProducts_grp_1',
       'NumOfProducts_grp_2', 'NumOfProducts_grp_3+'],
      dtype='str')

In [8]:
cols_a_exclure_lr = ["CreditScore", "Tenure", "EstimatedSalary", "HasCrCard"]
X_train_lr = X_train.drop(columns=cols_a_exclure_lr)
X_test_lr = X_test.drop(columns=cols_a_exclure_lr)

scaler = StandardScaler()
cols_a_scaler = ["Age", "Balance"]
X_train_lr[cols_a_scaler] = scaler.fit_transform(X_train_lr[cols_a_scaler])
X_test_lr[cols_a_scaler] = scaler.transform(X_test_lr[cols_a_scaler])

X_train_lr.describe()

,Age,Balance,IsActiveMember,Geography_France,Geography_Germany,Geography_Spain,NumOfProducts_grp_1,NumOfProducts_grp_2,NumOfProducts_grp_3+
count,8.000000e+03,8.000000e+03,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000
mean,2.486900e-16,5.773160e-17,0.514875,0.499125,0.250625,0.250250,0.508875,0.459000,0.032125
std,1.000063e+00,1.000063e+00,0.499810,0.500030,0.433400,0.433184,0.499952,0.498347,0.176343
min,-1.989948e+00,-1.226059e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-6.599355e-01,-1.226059e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-1.849311e-01,3.318547e-01,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
75%,4.800751e-01,8.226886e-01,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
max,5.040117e+00,2.600500e+00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [9]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
X_train_lr.to_csv("../data/processed/X_train_lr.csv", index=False)
X_test_lr.to_csv("../data/processed/X_test_lr.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

In [10]:
import joblib

joblib.dump(encoder_geo, "../models/encoder_geography.pkl")
joblib.dump(encoder_prod, "../models/encoder_products.pkl")
joblib.dump(scaler, "../models/scaler.pkl")

['../models/scaler.pkl']